In [ ]:
# SKKU PubMed Research Lineage — ONE CELL
# 1) SKKU affiliation 논문 수집
# 2) citation / 동일 SKKU 연구자 / MeSH-keyword 유사도 연결
# 3) ORCID 기반으로 연구자가 SKKU를 떠난 뒤의 후속 논문까지 추적
# 4) 결과 CSV/Markdown/HTML 생성 + 주요 표 표시

import os, sys, subprocess, json
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

# ===== 사용자 설정 =====
START_YEAR = 2010
END_YEAR = 2026
MAX_RESULTS = 200        # 먼저 200 권장. 전체 확대 시 500~1000
MAX_AUTHORS = 50         # ORCID 보유 SKKU 연구자 추적 수
MAX_PER_AUTHOR = 150
TOPIC = ""               # 예: "stroke OR cerebrovascular"; 전체면 ""
TOPIC_THRESHOLD = 0.30
NCBI_EMAIL = os.environ.get("NCBI_EMAIL") or input("NCBI email: ").strip()
NCBI_API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # 있으면 자동 사용

if not NCBI_EMAIL:
    raise ValueError("NCBI_EMAIL이 필요합니다.")

# ===== GitHub 최신 코드 동기화 =====
REPO = Path("/content/Paper_AI_Assistant")
URL = "https://github.com/kimtk94/Paper_AI_Assistant.git"

if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "-q", URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", "origin/main"], check=True)

os.chdir(REPO)

# ===== 실행 전 문법 검사 =====
for script in ["src/skku_pubmed_lineage.py", "src/skku_pubmed_annotation.py", "src/skku_pubmed_author_followup.py"]:
    subprocess.run([sys.executable, "-m", "py_compile", script], check=True)
print("✅ Python syntax check passed")

env = os.environ.copy()
env["NCBI_EMAIL"] = NCBI_EMAIL
if NCBI_API_KEY:
    env["NCBI_API_KEY"] = NCBI_API_KEY

# ===== STEP 1: SKKU PubMed graph =====
cmd1 = [
    sys.executable, "src/skku_pubmed_lineage.py",
    "--start-year", str(START_YEAR),
    "--end-year", str(END_YEAR),
    "--max-results", str(MAX_RESULTS),
    "--topic-threshold", str(TOPIC_THRESHOLD),
    "--output-dir", "outputs/skku_pubmed_lineage",
]
if TOPIC.strip():
    cmd1 += ["--topic", TOPIC.strip()]

print("\n=== STEP 1/2: SKKU PubMed graph ===")
subprocess.run(cmd1, check=True, env=env)

# ===== STEP 2: ORCID researcher continuation =====
cmd2 = [
    sys.executable, "src/skku_pubmed_author_followup.py",
    "--seed-json", "outputs/skku_pubmed_lineage/papers.json",
    "--start-year", "2002",
    "--end-year", str(END_YEAR),
    "--max-authors", str(MAX_AUTHORS),
    "--max-per-author", str(MAX_PER_AUTHOR),
    "--with-citations",
    "--max-citation-checks", "300",
    "--output-dir", "outputs/skku_pubmed_followup",
]

print("\n=== STEP 2/2: ORCID continuation ===")
subprocess.run(cmd2, check=True, env=env)

# ===== 빈 CSV도 안전하게 읽기 =====
def safe_csv(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

papers = safe_csv("outputs/skku_pubmed_lineage/papers.csv")
edges = safe_csv("outputs/skku_pubmed_lineage/edges.csv")
registry = safe_csv("outputs/skku_pubmed_followup/author_registry.csv")
followups = safe_csv("outputs/skku_pubmed_followup/lineage_papers.csv")
continuation = safe_csv("outputs/skku_pubmed_followup/continuation_edges.csv")
lineage_links = safe_csv("outputs/skku_pubmed_followup/lineage_edges.csv")
annotations = safe_csv("outputs/skku_pubmed_followup/paper_annotations.csv")

print("\n================ RESULT ================")
print(f"SKKU verified papers : {len(papers):,}")
print(f"SKKU graph edges     : {len(edges):,}")
print(f"Tracked researchers  : {len(registry):,}")
print(f"Follow-up papers     : {len(followups):,}")
print(f"Continuation edges   : {len(continuation):,}")
print(f"Research lineage     : {len(lineage_links):,}")
print(f"Annotated papers     : {len(annotations):,}")

if not edges.empty and "relation" in edges:
    print("\n[SKKU edge types]")
    display(edges["relation"].value_counts().rename_axis("relation").reset_index(name="count"))

if not registry.empty:
    print("\n[Tracked researchers]")
    cols = [c for c in ["name", "orcid", "confidence", "seed_pmids"] if c in registry.columns]
    display(registry[cols].head(30))

if not followups.empty:
    print("\n[Researcher publication continuation]")
    cols = [c for c in [
        "year", "tracked_authors", "title", "disease_terms", "methods",
        "data_types", "research_stage", "research_question",
        "journal", "skku_current", "pubmed_url"
    ] if c in followups.columns]
    display(followups[cols].sort_values(["tracked_authors", "year"], ascending=[True, True]).head(100))

if not annotations.empty:
    print("\n[Paper research profiles]")
    cols = [c for c in [
        "pmid", "disease_terms", "methods", "data_types",
        "research_stage", "research_question"
    ] if c in annotations.columns]
    display(annotations[cols].head(100))

if not lineage_links.empty:
    print("\n[Research lineage: direct citation > ORCID-only]")
    cols = [c for c in [
        "score", "relation", "source", "target", "tracked_authors",
        "source_stage", "target_stage", "progression", "evidence"
    ] if c in lineage_links.columns]
    display(lineage_links[cols].sort_values("score", ascending=False).head(100))

if not continuation.empty:
    print("\n[Raw continuation/citation edges]")
    display(continuation.head(100))

# ===== 결과 ZIP =====
zip_base = "/content/SKKU_PubMed_Lineage_results"
subprocess.run(
    ["zip", "-qr", zip_base + ".zip",
     "outputs/skku_pubmed_lineage", "outputs/skku_pubmed_followup"],
    check=True
)
print(f"\n📦 ZIP: {zip_base}.zip")
print("🌐 Interactive graph: /content/Paper_AI_Assistant/outputs/skku_pubmed_lineage/lineage.html")
print("📄 Research chains: /content/Paper_AI_Assistant/outputs/skku_pubmed_followup/continuation.md")

# Colab에서 interactive HTML 바로 표시
html_path = Path("outputs/skku_pubmed_lineage/lineage.html")
if html_path.exists():
    display(HTML("<b>완료:</b> 왼쪽 Files에서 <code>lineage.html</code> 또는 결과 ZIP을 열면 됩니다."))
